In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np 
import pandas as pd 

In [2]:
train =pd.read_csv("/kaggle/input/aification/train.csv") 
test = pd.read_csv("/kaggle/input/aification/test.csv")

In [3]:
train.head()

,id,bangla_question,english_question
0,1,25 kg ভরের বস্তুর গতিশক্তি 450 J হলে বেগ কত?,What is the velocity if the kinetic energy of ...
1,2,একটি বস্তু 40 m/s বেগে চলছে। 8 m/s² মন্দনে কত ...,An object is moving at 40 m/s velocity. In how...
2,3,একটি প্রোটনের ভর 9.11×10⁻³¹ kg হলে এর শক্তি কত...,If the mass of a proton is 9.11×10⁻³¹ kg; what...
3,4,একটি বস্তু 70 m/s বেগে চলছে। 2 m/s² মন্দনে কত ...,An object is moving at 70 m/s velocity. In how...
4,5,44 kg ভরের একটি গাড়ি 55 m/s বেগে চলছে এবং 18 ...,A car of mass 44 kg is moving at 55 m/s veloci...


In [4]:
train.tail()

,id,bangla_question,english_question
4995,4996,একটি সমান্তর অনুক্রমের প্রথম পদ 23 এবং সাধারণ ...,The first term of an arithmetic progression is...
4996,4997,59 g NaOH এবং 62 g HCl বিক্রিয়া করে NaCl উৎপন...,59 g NaOH and 62 g HCl react to produce NaCl. ...
4997,4998,একটি ইন্ডাকটরে ৬ H প্রতিবাদকতা এবং ৯০ Hz ফ্রিক...,An inductor has an inductance of 6 H and a fre...
4998,4999,একটি দ্বিঘাত সমীকরণ 3x² + -18x + -216 = 0 এর ব...,Find the real roots of the quadratic equation ...
4999,5000,322K তাপমাত্রার একটি সিস্টেম থেকে 3220 J তাপ শ...,If a system at 322K temperature absorbs 3220 J...


In [5]:
test.head()

,id,bangla_question
0,1,রক্তচাপ 115/80 mmHg হলে পালস প্রেসার কত?
1,2,150 J কাজ করে একটি বস্তুকে 20 m দূরত্বে স্থানা...
2,3,একটি আয়তক্ষেত্রের দৈর্ঘ্য 21 cm এবং প্রস্থ 30...
3,4,"DNA অণুতে যদি অ্যাডেনিন 36% থাকে, তাহলে থাইমিন..."
4,5,45 g CH₃COOH এ কতগুলি মোল পদার্থ আছে? (আণবিক ভ...


# Data Cleaning

In [6]:
import re
def clean(text):
  text = re.sub(r"\s+"," ", text).strip()
  return text

In [7]:
train['bangla_question'] = train['bangla_question'].apply(clean)
train['english_question'] = train['english_question'].apply(clean)

In [8]:
train.head()

,id,bangla_question,english_question
0,1,25 kg ভরের বস্তুর গতিশক্তি 450 J হলে বেগ কত?,What is the velocity if the kinetic energy of ...
1,2,একটি বস্তু 40 m/s বেগে চলছে। 8 m/s² মন্দনে কত ...,An object is moving at 40 m/s velocity. In how...
2,3,একটি প্রোটনের ভর 9.11×10⁻³¹ kg হলে এর শক্তি কত...,If the mass of a proton is 9.11×10⁻³¹ kg; what...
3,4,একটি বস্তু 70 m/s বেগে চলছে। 2 m/s² মন্দনে কত ...,An object is moving at 70 m/s velocity. In how...
4,5,44 kg ভরের একটি গাড়ি 55 m/s বেগে চলছে এবং 18 ...,A car of mass 44 kg is moving at 55 m/s veloci...


In [9]:
#remove rows with missing translation
train = train.dropna(subset = ["bangla_question", "english_question"])

train["bangla_question"] = train["bangla_question"].astype(str)
train["english_question"] = train["english_question"].astype(str)

In [10]:
#remove very small strings
train = train[train["bangla_question"].str.len() > 5]
train = train[train["english_question"].str.len() > 5]

# Split into Train / Validation (dev)

In [11]:
train = train.sample(frac = 1, random_state = 42).reset_index(drop = True)
train_set = train.iloc[:4_000]
dev_set = train.iloc[4_000:5_000]

train_set[["english_question", "bangla_question"]].to_csv("train.tsv", sep = "\t", index = False, header = False)
dev_set[["english_question", "bangla_question"]].to_csv("dev.tsv", sep = "\t", index = False, header = False) 

In [12]:
!cat train.tsv dev.tsv > combined.txt

# SentencePiece BPE Tokenizer

In [13]:
import sentencepiece as spm
spm.SentencePieceTrainer.train(
    input = "combined.txt",
    model_prefix = "en_bn_spm",
    vocab_size = 8_000,
    model_type = "bpe",
    character_coverage = 1.0,
    unk_id = 0, pad_id = 1, bos_id = 2, eos_id = 3,
    user_defined_symbols="<en>,<bn>",
    normalization_rule_name="nmt_nfkc",
    split_digits=True,
    byte_fallback=True
)

print("Tokenizer trained to en_bn_spm.model")

Tokenizer trained to en_bn_spm.model


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: combined.txt
  input_format: 
  model_prefix: en_bn_spm
  model_type: BPE
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  user_defined_symbols: <en>
  user_defined_symbols: <bn>
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 2
  eos_id: 3
  pad_id: 1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surf

# HF Dataset + Tokenization

## Load NLLB-200 distilled (600 M)

In [14]:
!pip install -U transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 74.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 1.0.0rc2
    Uninstalling huggingface-hub-1.0.0rc2:
      Successfully uninstalled huggingface-hub-1.0.0rc2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.2
    Uninstalling tokenizers-0.21.2:
      Successfully uninstalled tokenizers-0.21.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.53.3
    Uninstalling transformers-4.53.3:
      Successfully uninstalled transformers-4.53.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source 

In [15]:
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    src_lang="eng_Latn",
    tgt_lang="ben_Beng",
    use_fast=False,
    use_default_chat_template=False,
)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [16]:
#Load tsv
data_files = {"train": "train.tsv", "validation": "dev.tsv"}
raw = load_dataset(
    "csv",
    data_files = data_files,
    delimiter = "\t",
    column_names = ["en", "bn"]
)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [17]:
def preprocess(ex):
    #src = bengali, target = english
    inputs = tokenizer(
        ex["bn"],
        max_length = 128,
        truncation = True
    )
    with tokenizer.as_target_tokenizer():
        targets = tokenizer(
            ex["en"],
            max_length = 128,
            truncation = True
        )
    inputs["labels"] = targets["input_ids"]
    return inputs

tokenized = raw.map(
    preprocess,
    batched = True,
    remove_columns = ["en","bn"]
)

print(tokenized)

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 4000
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
})


# Fine-tune NLLB

In [18]:
#model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")
model.resize_token_embeddings(len(tokenizer))

M2M100ScaledWordEmbedding(256204, 1024, padding_idx=1)

In [19]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    padding = True
)

args = Seq2SeqTrainingArguments(
    output_dir = "./bn_en_nllb",
    eval_strategy = "epoch",
    learning_rate = 5e-5,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=2,           # change to 3 later
    predict_with_generate=True,
    fp16=True,
    logging_steps=200,
    report_to="none",
    save_strategy="epoch",
    load_best_model_at_end=True
)

trainer = Seq2SeqTrainer(
    model = model,
    args = args,
    train_dataset = tokenized["train"],
    eval_dataset = tokenized["validation"],
    tokenizer = tokenizer,
    data_collator = data_collator
)

# Final Trainerrrrrrrrr 🎉🎉

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.286100,0.114946
2,0.127000,0.096532


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=500, training_loss=0.18503855514526368, metrics={'train_runtime': 891.5073, 'train_samples_per_second': 8.974, 'train_steps_per_second': 0.561, 'total_flos': 1366562617688064.0, 'train_loss': 0.18503855514526368, 'epoch': 2.0})

# Translate Test set

In [21]:
english_token = "eng_Latn"
if english_token in tokenizer.get_vocab():
    english_token_id = tokenizer.convert_tokens_to_ids(english_token)
    print(f"Found {english_token} -> ID: {english_token_id}")
else:
    raise ValueError("English token 'eng_Latn' not in tokenizer vocabulary!")

Found eng_Latn -> ID: 256047


In [22]:
test["english_question"] = "" #placeholder
def translate_batch(batch):
    inputs = tokenizer(
        batch["bangla_question"],
        return_tensors="pt",
        padding=True,
        truncation=True, 
        max_length=128
    ).to(model.device)

    generated = model.generate(
        **inputs,
        forced_bos_token_id=english_token_id,
        max_length=128,
        num_beams=5
    )

    decoded_translation = tokenizer.decode(generated[0], skip_special_tokens=True)
    batch["english_question"] = decoded_translation
    
    return batch

test = test.apply(translate_batch, axis=1)

In [23]:
test.head()

,id,bangla_question,english_question
0,1,রক্তচাপ 115/80 mmHg হলে পালস প্রেসার কত?,What is the pulse pressure at 115/80 mmHg?
1,2,150 J কাজ করে একটি বস্তুকে 20 m দূরত্বে স্থানা...,How much newtonian force is required to move a...
2,3,একটি আয়তক্ষেত্রের দৈর্ঘ্য 21 cm এবং প্রস্থ 30...,If the length of a rectangle is 21 cm and the ...
3,4,"DNA অণুতে যদি অ্যাডেনিন 36% থাকে, তাহলে থাইমিন...",If 36% adenin is present in a DNA molecule; wh...
4,5,45 g CH₃COOH এ কতগুলি মোল পদার্থ আছে? (আণবিক ভ...,How many moles of matter are there in 45 g CH3...


In [24]:
test[["id","english_question"]].to_csv("submission.csv", index = False)
print("Submission.csv ready for ZeroAuth 🎉")

Submission.csv ready for ZeroAuth 🎉
